In [1]:
!pip -q install \
transformers \
timm \
sentencepiece \
accelerate \
datasets \
scikit-learn \
pandas \
pillow

In [2]:
import os
import copy
import random
import numpy as np
import pandas as pd

from PIL import Image

import torch
import torch.nn as nn
from torchvision import transforms

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score, confusion_matrix, ConfusionMatrixDisplay

from transformers import (
    AutoModel,
    AutoTokenizer,
    AutoImageProcessor,
    SiglipVisionModel,
    get_cosine_schedule_with_warmup
)
from torch.amp import GradScaler

device = torch.device("cuda:1")
print(device)

cuda:1


In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [4]:
TRAIN_IMAGES = "/kaggle/input/datasets/anikaaaakjnkj/meme-dataset/train_images"
TRAIN_CSV = "/kaggle/input/datasets/anikaaaakjnkj/meme-dataset/train_labels.csv"

IMAGE_MODEL = "google/siglip-base-patch16-512"
TEXT_MODEL = "xlm-roberta-base" 

In [5]:
df = pd.read_csv(TRAIN_CSV)

df = df.rename(columns={
    "Image_id": "image",
    "Text": "text",
    "Level1": "label1",
    "Level2": "label2"
})

df["text"] = df["text"].fillna("").astype(str).str.strip()

df.head()

,image,text,label1,label2
0,0,• கர்நாடகாவில் ரஜினியின் காலா படம் வெளியா முதல...,Troll/Oppose,Troll/Oppose Against Person
1,1,Thalapathy Vijay Thiruchchi Varugai...\nThiruc...,Support/Praise,Support for person
2,2,"""அண்ண சொல்லுங்க.\n#GET OU MODI\n#GET OUT STALI...",Troll/Oppose,Troll/Oppose Against Party
3,3,சீமான் கவுண்டமணி மரண கலாய்\nதேங்காய அங்கேயே வை...,Troll/Oppose,Troll/Oppose Against Person
4,4,TOI - LINE OF NO CONTROL \nKAMAL HAASAN START...,Support/Praise,Support for party


In [6]:
level1_encoder = LabelEncoder()
level2_encoder = LabelEncoder()

df["label1_id"] = level1_encoder.fit_transform(df.label1)
df["label2_id"] = level2_encoder.fit_transform(df.label2)

print("Level 1 classes:", list(level1_encoder.classes_))
print("Level 2 classes:", list(level2_encoder.classes_))

# Target head: Party vs Person, derived from label2 (this was defined before
# but never actually wired into training -- now it feeds the target_head below)
entity_map = {
    "Support for party": 0,
    "Support for person": 1,
    "Troll/Oppose Against Party": 0,
    "Troll/Oppose Against Person": 1
}
df["entity_id"] = df["label2"].map(entity_map)

assert df["entity_id"].isna().sum() == 0, "entity_map didn't cover every label2 value -- check spelling"


Level 1 classes: ['Support/Praise', 'Troll/Oppose']
Level 2 classes: ['Support for party', 'Support for person', 'Troll/Oppose Against Party', 'Troll/Oppose Against Person']


In [7]:
image_processor = AutoImageProcessor.from_pretrained(IMAGE_MODEL)
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL)

train_transform = transforms.Compose([
    # transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(3),
    transforms.ColorJitter(brightness=0.10, contrast=0.10, saturation=0.10)
])

val_transform = None

preprocessor_config.json:   0%|          | 0.00/368 [00:00<?, ?B/s]

The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
img = Image.open(os.path.join(TRAIN_IMAGES, "001.jpg")).convert("RGB")

print("Original:", img.size)

processed = image_processor(img, return_tensors="pt")

print("Processed:", processed["pixel_values"].shape)

Original: (1350, 1688)
Processed: torch.Size([1, 3, 512, 512])


In [9]:
class MemeDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_name = f"{int(row.image):03d}.jpg"
        image = Image.open(os.path.join(TRAIN_IMAGES, image_name)).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return {
            "image": image,
            "text": row.text,
            "label1": row.label1_id,   # Support/Troll (2-way)
            "target": row.entity_id,   # Party/Person   (2-way)
            "label2": row.label2_id,   # fine-grained    (4-way, main task)
        }


def collate_fn(batch):
    images = [x["image"] for x in batch]
    image_inputs = image_processor(images, return_tensors="pt")

    texts = [x["text"] for x in batch]
    text_inputs = tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=512,
        return_tensors="pt"
    )

    return {
        "pixel_values": image_inputs["pixel_values"],
        "input_ids": text_inputs["input_ids"],
        "attention_mask": text_inputs["attention_mask"],
        "label1": torch.tensor([x["label1"] for x in batch]),
        "target": torch.tensor([x["target"] for x in batch]),
        "label2": torch.tensor([x["label2"] for x in batch]),
    }

In [10]:
class CrossAttentionFusion(nn.Module):

    def __init__(self, dim=768, heads=8):
        super().__init__()
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=heads,
            batch_first=True
        )
        self.norm = nn.LayerNorm(dim)

    def forward(self, text_tokens, image_tokens):
        attended, _ = self.cross_attention(
            query=text_tokens,
            key=image_tokens,
            value=image_tokens
        )
        return self.norm(attended + text_tokens)


class MemeClassifier(nn.Module):

    def __init__(self, image_encoder, text_encoder):
        super().__init__()

        self.image_encoder = image_encoder
        self.text_encoder = text_encoder

        self.cross = CrossAttentionFusion(dim=768, heads=8)

        self.classifier = nn.Sequential(
            nn.Linear(1536, 1024),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(1024, 512),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.GELU()
        )

        # independent branch for the target (Party/Person) task -- separate
        # capacity so it isn't forced through the same bottleneck as level1
        self.target_branch = nn.Sequential(
            nn.Linear(1536, 512),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.GELU()
        )

        self.level1_head = nn.Linear(256, 2)
        self.target_head = nn.Linear(256, 2)
        # +2 +2 for the level1/target logits concatenated in below
        self.level2_head = nn.Linear(256 + 2 + 2, 4)

    def forward(
    self,
    pixel_values,
    input_ids,
    attention_mask,
    image_ablation=False,
    text_ablation=False,):

        image_outputs = self.image_encoder(pixel_values=pixel_values)
        image_tokens = image_outputs.last_hidden_state

        text_outputs = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        text_tokens = text_outputs.last_hidden_state

        image_tokens_for_cross = torch.zeros_like(image_tokens) if image_ablation else image_tokens
        attended = self.cross(text_tokens, image_tokens_for_cross)

        text_feature = attended.mean(dim=1)
        image_feature = image_outputs.pooler_output

        if image_ablation:
            image_feature = torch.zeros_like(image_feature)

        if text_ablation:
            text_feature = torch.zeros_like(text_feature)

        raw_fused = torch.cat([text_feature, image_feature], dim=1)
        
        fused = self.classifier(raw_fused)
        
        target_feature = self.target_branch(raw_fused)
        
        level1_logits = self.level1_head(fused)
        target_logits = self.target_head(target_feature)

        level2_input = torch.cat(
            [fused, level1_logits, target_logits],
            dim=1
        )
        
        level2_logits = self.level2_head(level2_input)

        return {
            "level1": level1_logits,
            "target": target_logits,
            "level2": level2_logits,
        }


In [11]:
def build_model():

    image_encoder = SiglipVisionModel.from_pretrained(IMAGE_MODEL)
    text_encoder = AutoModel.from_pretrained(TEXT_MODEL)

    for p in text_encoder.parameters():
        p.requires_grad = False
    for layer in text_encoder.encoder.layer[-6:]:
        for p in layer.parameters():
            p.requires_grad = True

    for p in image_encoder.parameters():
        p.requires_grad = False
    for layer in image_encoder.vision_model.encoder.layers[-2:]:
        for p in layer.parameters():
            p.requires_grad = True

    model = MemeClassifier(image_encoder, text_encoder).to(device)
    return model


def build_optimizer_scheduler(model, train_loader, epochs):
    optimizer = torch.optim.AdamW(
        [
            {"params": model.image_encoder.parameters(), "lr": 1e-5},
            {"params": model.text_encoder.parameters(), "lr": 1e-5},
            {"params": model.cross.parameters(), "lr": 3e-5},
            {"params": model.classifier.parameters(), "lr": 3e-5},
            {"params": model.target_branch.parameters(), "lr": 3e-5},
            {"params": model.level1_head.parameters(), "lr": 3e-5},
            {"params": model.target_head.parameters(), "lr": 3e-5},
            {"params": model.level2_head.parameters(), "lr": 3e-5},
        ],
        weight_decay=0.01,
        betas=(0.9, 0.999)
    )

    total_steps = len(train_loader) * epochs
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps
    )

    return optimizer, scheduler


def class_weights_for(train_df, col, n_classes, device):
    counts = (
        train_df[col].value_counts()
        .reindex(range(n_classes), fill_value=1)
        .sort_index()
        .values
    )
    w = counts.sum() / counts
    w = w / w.mean()
    return torch.tensor(w, dtype=torch.float32, device=device)


def build_criteria(train_df):
    crit_level1 = nn.CrossEntropyLoss(
        weight=class_weights_for(train_df, "label1_id", 2, device),
        label_smoothing=0.05
    )
    crit_target = nn.CrossEntropyLoss(
        weight=class_weights_for(train_df, "entity_id", 2, device),
        label_smoothing=0.05
    )
    crit_level2 = nn.CrossEntropyLoss(
        weight=class_weights_for(train_df, "label2_id", 4, device),
        label_smoothing=0.05
    )
    return crit_level1, crit_target, crit_level2

LAMBDA_LEVEL1 = 0.3
LAMBDA_TARGET = 0.8 
LAMBDA_LEVEL2 = 1.0

In [12]:
def train_one_epoch(model, loader, optimizer, scheduler, scaler, criteria):
    crit_l1, crit_target, crit_l2 = criteria

    model.train()
    running_loss = 0

    preds1, labels1 = [], []
    preds_target, labels_target = [], []
    preds2, labels2 = [], []

    for batch in loader:
        pixel_values = batch["pixel_values"].to(device)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        label1 = batch["label1"].to(device)
        target = batch["target"].to(device)
        label2 = batch["label2"].to(device)

        optimizer.zero_grad()

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            out = model(
    pixel_values,
    input_ids,
    attention_mask
)
            loss_l1 = crit_l1(out["level1"], label1)
            loss_target = crit_target(out["target"], target)
            loss_l2 = crit_l2(out["level2"], label2)

            loss = (
                LAMBDA_LEVEL1 * loss_l1 +
                LAMBDA_TARGET * loss_target +
                LAMBDA_LEVEL2 * loss_l2
            )

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running_loss += loss.item()
        preds1.extend(out["level1"].argmax(1).cpu().numpy())
        labels1.extend(label1.cpu().numpy())

        preds_target.extend(out["target"].argmax(1).cpu().numpy())
        labels_target.extend(target.cpu().numpy())

        preds2.extend(out["level2"].argmax(1).cpu().numpy())
        labels2.extend(label2.cpu().numpy())

    level1_f1 = f1_score(labels1, preds1, average="macro")
    target_f1 = f1_score(labels_target, preds_target, average="macro")
    level2_f1 = f1_score(labels2, preds2, average="macro")

    return (
        running_loss / len(loader),
        level1_f1,
        target_f1,
        level2_f1
    )


@torch.no_grad()
def validate(
    model,
    loader,
    criteria,
    image_ablation=False,
    text_ablation=False,
    verbose=False):
    crit_l1, crit_target, crit_l2 = criteria

    model.eval()
    running_loss = 0

    preds1, labels1 = [], []
    preds_target, labels_target = [], []
    preds2, labels2 = [], []

    for batch in loader:
        pixel_values = batch["pixel_values"].to(device)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        label1 = batch["label1"].to(device)
        target = batch["target"].to(device)
        label2 = batch["label2"].to(device)

        out = model(
    pixel_values,
    input_ids,
    attention_mask,
    image_ablation=image_ablation,
    text_ablation=text_ablation,
)

        loss_l1 = crit_l1(out["level1"], label1)
        loss_target = crit_target(out["target"], target)
        loss_l2 = crit_l2(out["level2"], label2)

        loss = (
            LAMBDA_LEVEL1 * loss_l1 +
            LAMBDA_TARGET * loss_target +
            LAMBDA_LEVEL2 * loss_l2
        )

        running_loss += loss.item()
        preds1.extend(out["level1"].argmax(1).cpu().numpy())
        labels1.extend(label1.cpu().numpy())

        preds_target.extend(out["target"].argmax(1).cpu().numpy())
        labels_target.extend(target.cpu().numpy())

        preds2.extend(out["level2"].argmax(1).cpu().numpy())
        labels2.extend(label2.cpu().numpy())

    level1_f1 = f1_score(labels1, preds1, average="macro")
    target_f1 = f1_score(labels_target, preds_target, average="macro")
    level2_f1 = f1_score(labels2, preds2, average="macro")

    if verbose:
        print(classification_report(labels2, preds2, target_names=level2_encoder.classes_))

    return (
    running_loss / len(loader),
    level1_f1,
    target_f1,
    level2_f1,
    labels2,
    preds2,
    labels_target,  
    preds_target
)

In [13]:
K_FOLDS = 5
EPOCHS_PER_FOLD = 20
PATIENCE = 4
BATCH_SIZE = 8

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)

fold_scores = []
oof_labels = np.full(len(df), -1)
oof_preds = np.full(len(df), -1)
oof_target_labels = np.full(len(df), -1)
oof_target_preds = np.full(len(df), -1)

ablation_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(df, df["label2_id"])):

    print("=" * 70)
    print(f"FOLD {fold + 1}/{K_FOLDS}")
    print("=" * 70)

    train_df_fold = df.iloc[train_idx].reset_index(drop=True)
    val_df_fold = df.iloc[val_idx].reset_index(drop=True)

    train_dataset = MemeDataset(train_df_fold, transform=train_transform)
    val_dataset = MemeDataset(val_df_fold, transform=val_transform)

    class_counts = train_df_fold["label2_id"].value_counts().reindex(range(4), fill_value=1)
    sample_weights = train_df_fold["label2_id"].map(1.0 / class_counts).values
    sampler = torch.utils.data.WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        sampler=sampler,           
        collate_fn=collate_fn,
        num_workers=2
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=2
    )

    model = build_model()
    optimizer, scheduler = build_optimizer_scheduler(model, train_loader, EPOCHS_PER_FOLD)
    scaler = GradScaler("cuda")
    criteria = build_criteria(train_df_fold)

    best_score = 0
    counter = 0
    best_state = None
    best_val_labels, best_val_preds = None, None

    for epoch in range(EPOCHS_PER_FOLD):
        train_loss, train_l1_f1, train_target_f1, train_l2_f1 = train_one_epoch(
            model,
            train_loader,
            optimizer,
            scheduler,
            scaler,
            criteria
        )

        val_loss, val_l1_f1, val_target_f1, val_l2_f1, val_labels, val_preds, val_target_labels, val_target_preds = validate(
            model,
            val_loader,
            criteria
        )

        print(
            f"Epoch {epoch+1:2d}"
            f" | Train L1 {train_l1_f1:.3f}"
            f" | Train Target {train_target_f1:.3f}"
            f" | Train L2 {train_l2_f1:.3f}"
            f" || Val L1 {val_l1_f1:.3f}"
            f" | Val Target {val_target_f1:.3f}"
            f" | Val L2 {val_l2_f1:.3f}"
        )

        if val_l2_f1 > best_score:
            best_score = val_l2_f1
            counter = 0
            best_state = copy.deepcopy(model.state_dict())
            best_val_labels, best_val_preds = val_labels, val_preds
            best_val_target_labels, best_val_target_preds = val_target_labels, val_target_preds
        else:
            counter += 1
            if counter >= PATIENCE:
                print("Early stopping")
                break

        # Restore the best model
    model.load_state_dict(best_state)
    model.eval()
    
    print("="*60)
    print("Running ablation study")
    print("="*60)

    # Normal
    normal = validate(
        model,
        val_loader,
        criteria
    )

    # Text only (remove image)
    text_only = validate(
        model,
        val_loader,
        criteria,
        image_ablation=True
    )

    # Image only (remove text)
    image_only = validate(
        model,
        val_loader,
        criteria,
        text_ablation=True
    )

    print(f"{'':12s} {'L1 F1':>8s} {'Target F1':>10s} {'L2 F1':>8s}")
    print(f"{'Normal':12s} {normal[1]:8.4f} {normal[2]:10.4f} {normal[3]:8.4f}")
    print(f"{'Text Only':12s} {text_only[1]:8.4f} {text_only[2]:10.4f} {text_only[3]:8.4f}")
    print(f"{'Image Only':12s} {image_only[1]:8.4f} {image_only[2]:10.4f} {image_only[3]:8.4f}")

    ablation_results.append({
        "fold": fold,
        "normal_l1": normal[1], "normal_target": normal[2], "normal_l2": normal[3],
        "text_only_l1": text_only[1], "text_only_target": text_only[2], "text_only_l2": text_only[3],
        "image_only_l1": image_only[1], "image_only_target": image_only[2], "image_only_l2": image_only[3],
    })

    torch.save(best_state, f"best_model_fold{fold}.pt")
    fold_scores.append(best_score)
    
    oof_labels[val_idx] = best_val_labels
    oof_preds[val_idx] = best_val_preds
    oof_target_labels[val_idx] = best_val_target_labels
    oof_target_preds[val_idx] = best_val_target_preds
    
    print(f"Fold {fold + 1} best val macro-F1: {best_score:.4f}\n")
    
    del model, optimizer, scheduler
    torch.cuda.empty_cache()

fold_scores = np.array(fold_scores)
print("=" * 70)
print(f"K-Fold results: {fold_scores.round(4)}")
print(f"Mean macro-F1: {fold_scores.mean():.4f}  |  Std: {fold_scores.std():.4f}")
print("=" * 70)

# --- Ablation summary across all folds ---
ablation_df = pd.DataFrame(ablation_results)
print("\nAblation study, mean +/- std across folds:")
for head in ["l1", "target", "l2"]:
    print(f"\n  {head.upper()}:")
    for cond in ["normal", "text_only", "image_only"]:
        col = f"{cond}_{head}"
        print(f"    {cond:12s} {ablation_df[col].mean():.4f} +/- {ablation_df[col].std():.4f}")
ablation_df

FOLD 1/5


config.json:   0%|          | 0.00/322 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/815M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

SiglipVisionModel LOAD REPORT from: google/siglip-base-patch16-512
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
logit_scale                                                  | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.b

...

SiglipVisionModel LOAD REPORT from: google/siglip-base-patch16-512
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
logit_scale                                                  | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.b

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch  1 | Train L1 0.329 | Train Target 0.351 | Train L2 0.119 || Val L1 0.225 | Val Target 0.202 | Val L2 0.045
Epoch  2 | Train L1 0.333 | Train Target 0.346 | Train L2 0.122 || Val L1 0.225 | Val Target 0.202 | Val L2 0.145
Epoch  3 | Train L1 0.574 | Train Target 0.333 | Train L2 0.275 || Val L1 0.779 | Val Target 0.210 | Val L2 0.241
Epoch  4 | Train L1 0.888 | Train Target 0.674 | Train L2 0.485 || Val L1 0.763 | Val Target 0.678 | Val L2 0.369
Epoch  5 | Train L1 0.925 | Train Target 0.842 | Train L2 0.729 || Val L1 0.796 | Val Target 0.616 | Val L2 0.516
Epoch  6 | Train L1 0.933 | Train Target 0.874 | Train L2 0.775 || Val L1 0.835 | Val Target 0.710 | Val L2 0.622
Epoch  7 | Train L1 0.952 | Train Target 0.912 | Train L2 0.863 || Val L1 0.882 | Val Target 0.712 | Val L2 0.624
Epoch  8 | Train L1 0.973 | Train Target 0.959 | Train L2 0.936 || Val L1 0.872 | Val Target 0.704 | Val L2 0.650
Epoch  9 | Train L1 0.982 | Train Target 0.969 | Train L2 0.949 || Val L1 0.851 | Val Ta

,fold,normal_l1,normal_target,normal_l2,text_only_l1,text_only_target,text_only_l2,image_only_l1,image_only_target,image_only_l2
0,0,0.892738,0.722553,0.662643,0.224000,0.613209,0.200461,0.853682,0.224437,0.400163
1,1,0.892738,0.770972,0.676529,0.253255,0.740399,0.204996,0.812560,0.213514,0.337336
2,2,0.837868,0.740399,0.671854,0.217742,0.667142,0.234483,0.777854,0.224437,0.312571
3,3,0.890312,0.658049,0.611868,0.389341,0.602510,0.227365,0.892395,0.421697,0.372742
4,4,0.905160,0.752101,0.710955,0.224900,0.685545,0.209811,0.899374,0.282090,0.332171
